<a href="https://colab.research.google.com/github/nepslor/Teaching/blob/main/CAS_BDML/W11/ML_models_and_chronos_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced models for forecasting
In this exercise we will see how to run advanced forecasting models
* A lightgboost model
* Chronos 2 model


In [ ]:
!pip install lightgbm wget

In [ ]:

import wget
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Setup path
save_folder = 'jena_climate'
if not os.path.exists(save_folder):
    os.makedirs(save_folder)

# Download Data
print("Downloading data...")
dat = wget.download('https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip', out=save_folder)

# Extract
with zipfile.ZipFile(dat, 'r') as zip_ref:
    zip_ref.extractall(save_folder)

# Parse Dates
dateparse = lambda x: datetime.strptime(x, '%d.%m.%Y %H:%M:%S')
csv_path = os.path.join(save_folder, 'jena_climate_2009_2016.csv')

# Load & Resample
print("\nProcessing Dataframe...")
df_weather = pd.read_csv(csv_path, parse_dates=['Date Time'], index_col='Date Time', date_parser=dateparse)
# Resample to hourly to make the data manageable
df_weather = df_weather.resample('1h').mean()

# Inspect
print(f"Data loaded. Shape: {df_weather.shape}")
df_weather.head(3)

In [ ]:
original_cols = df_weather.columns

In [ ]:
# @title 1. Data Visualization
print('#'*50)
print('Shape of dataset: {}'.format(df_weather.shape))
print('#'*50)
print('Missing vals: {}'.format(df_weather.isna().sum()))
print('#'*50)
df_weather.iloc[:2000, :].plot(subplots=True, figsize=(20, 10))
plt.show()


In [ ]:
df_weather['T (degC)'].plot(figsize=(20, 4))
df_weather.plot(figsize=(20, 4), linewidth=0.1)
plt.ylim(-100, 360)

In [ ]:
# plot distributions of variables
df_weather.hist(figsize=(20, 10), bins=100)
plt.show()


# Outlier detection for time series
In the previous plot, we see some extreme values are present in the time series. We try to filter them using two mechanisms: 
1. A standard z-score style threshold on absolute distance from the series mean
2. A time-series specific technique: we identify as outliers points exceeding a threshold on short-term deviations (deviations in a saliency score)
 
Detected outliers are set to NaN and then smoothly interpolated with PCHIP so downstream forecasting models are not dominated by spurious spikes.


In [ ]:
ma = df_weather.rolling(window=24*7, center=True, min_periods=1).mean()
m_std = df_weather.rolling(window=24*7, center=True, min_periods=1).std()
saliency = (df_weather-ma) / m_std
saliency.plot(figsize=(20, 4));


In [ ]:
outlier = (saliency.abs() > saliency.std()*5) | ((df_weather - df_weather.mean()).abs() > df_weather.std() * 10)

df_weather_noo = df_weather.copy()
# set outliers to nan, interpolate
for variable_key in df_weather.columns:
  df_weather_noo.loc[outlier[variable_key], variable_key] = np.nan
  df_weather_noo[variable_key] = df_weather_noo[variable_key].interpolate(method='pchip')


for variable_key in df_weather.columns:
  outlier_days = np.unique([pd.to_datetime(d.date())  for d  in outlier[variable_key].loc[outlier[variable_key]].index])
  if len(outlier_days) == 0:
    continue
  fig, ax = plt.subplots(len(outlier_days), 1, figsize=(15, len(outlier_days)), layout='constrained')
  ax = np.atleast_1d(ax)
  for a, od in zip(ax.ravel(), outlier_days):
    df_weather[variable_key].loc[df_weather.index>=od].iloc[:200].plot(ax=a)
    df_weather_noo[variable_key].loc[df_weather_noo.index>=od].iloc[:200].plot(ax=a, linestyle='--')
    o_idx = outlier[variable_key].loc[outlier.index>=od].iloc[:200]
    o_idx = o_idx[o_idx].index
    a.scatter(o_idx, df_weather[variable_key].loc[o_idx], color='red')
    a.set_xlabel('')
    a.set_xticks([])
    a.set_xticklabels([])
    a.tick_params(axis='x', bottom=False, labelbottom=False)
    # More aggressive tick removal
    a.xaxis.set_major_locator(plt.NullLocator())
    a.xaxis.set_minor_locator(plt.NullLocator())
    a.text(y=0.9, x=0.9, s=od.strftime('%Y-%m-%d'), ha='right', va='top', transform=a.transAxes, fontsize=12)
  ax[0].set_title(variable_key)


In [ ]:
df_weather_noo.plot(figsize=(20, 4))

In [ ]:
df_weather = df_weather_noo

# Forecasting 'T (degC)'
We want to forecast the next day of T (degC) using a direct strategy. 
Thus, we're gonna fit 24 different models, and compare two regressors:
1. linear models
2. lightGBM models

We start by crafting some features, based on the ACF plot of the signal

In [ ]:
# retrieve ACF plots for T (deg C)
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(figsize=(10, 2))
plot_acf(df_weather['T (degC)'], lags=24*7, ax=ax)
plt.show()


In [ ]:
# @title 2. Feature generation and embargo split
target = 'T (degC)'
def add_features(df, target = 'T (degC)'):
  df_weather = df.copy()

  df_weather['hour'] = df_weather.index.hour
  df_weather['day_of_week'] = df_weather.index.dayofweek
  df_weather['day_of_year'] = df_weather.index.dayofyear
  df_weather['month'] = df_weather.index.month
  df_weather['year'] = df_weather.index.year

  # Calculate the rolling average of 'T (degC)'
  df_weather[f'{target}_rolling_avg_24h'] = df_weather[target].rolling(window=24, min_periods=1).mean()
  df_weather[f'{target}_rolling_avg_48h'] = df_weather[target].rolling(window=48, min_periods=1).mean()
  df_weather[f'{target}_rolling_avg_72h'] = df_weather[target].rolling(window=72, min_periods=1).mean()
  df_weather[f'{target}_daily_mean_7d'] = df_weather[target].rolling(window=24*7, min_periods=1).mean()
  df_weather[f'{target}_daily_max_7d'] = df_weather[target].rolling(window=24*7, min_periods=1).max()
  df_weather[f'{target}_daily_min_7d'] = df_weather[target].rolling(window=24*7, min_periods=1).min()

  # Create 48 lagged features for 'T (degC)'
  history = {}
  for i in range(1, 24):
      history[f'{target}_lag_{i}'] = df_weather[target].shift(i)



  # Create 24 targets 'T (degC)'
  future = {}
  for i in range(1, 49):
      future[f'{target}_lag_{-i}'] = df_weather[target].shift(-i)

  df_weather = pd.concat([df_weather, pd.DataFrame(history), pd.DataFrame(future)], axis=1)
  df_weather.dropna(inplace=True)
  return df_weather

df_weather = add_features(df_weather_noo, target)
print("Generated new features. Displaying first 5 rows with new columns:")
print(f"New shape of df_weather: {df_weather.shape}")
df_weather.head()


## Embargo split
We keep a temporal embargo between calibration and test windows to reduce leakage from adjacent timestamps in highly autocorrelated data.
This setup creates three chronological blocks (train, calibration, test) and leaves a gap of 3 days before the test period.


In [ ]:
# @title 3 - Train-test split with embargo
embargo_period = 24*10
n_te = 24*365
n_cal = 24*365

# Chronological split: train | calibration | EMBARGO | test
df_tr = df_weather.iloc[:-(n_te + n_cal + embargo_period)]
df_cal = df_weather.iloc[-(n_te + n_cal + embargo_period):-(n_te + embargo_period)]
df_te = df_weather.iloc[-n_te:]

target_names = [c for c in df_weather.columns if target in c and 'lag_-' in c]
x_tr, y_tr = df_tr.drop(columns=target_names), df_tr[target_names]
x_cal, y_cal = df_cal.drop(columns=target_names), df_cal[target_names]
x_te, y_te = df_te.drop(columns=target_names), df_te[target_names]

print('Train end:', df_tr.index.max())
print('Cal end:', df_cal.index.max())
print('Test start:', df_te.index.min())
print('Embargo hours between cal end and test start:', int((df_te.index.min() - df_cal.index.max()) / pd.Timedelta(hours=1)) - 1)


In [ ]:
# lineplot of train, validation, and test periods with different colors
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_tr.index, y=df_tr[target],
    mode='lines',
    name='Train',
    line=dict(color='royalblue', width=1)
))

fig.add_trace(go.Scatter(
    x=df_cal.index, y=df_cal[target],
    mode='lines',
    name='Validation',
    line=dict(color='orange', width=1)
))

fig.add_trace(go.Scatter(
    x=df_te.index, y=df_te[target],
    mode='lines',
    name='Test',
    line=dict(color='green', width=1)
))

fig.update_layout(
    title=f'{target} split by period (train/validation/test)',
    xaxis_title='Date Time',
    yaxis_title=target,
    width=1100,
    height=420,
    margin=dict(l=40, r=20, t=50, b=40),
)

fig.show()


In [ ]:
from functools import partial
from tqdm import tqdm
from lightgbm import LGBMRegressor
from sklearn.linear_model import LinearRegression
from statsforecast.models import SeasonalExponentialSmoothingOptimized

def create_pdf(y_cal, y_hat_cal, q_vect = np.arange(11)/10):
  err = y_cal - y_hat_cal
  q_errs = np.quantile(err, q_vect, axis=0).T
  return q_errs


class Multi_reg:
  def __init__(self, model_class) -> None:
     self.model_class = model_class
     self.models = []

  def fit(self, x, y, budget=10000):
    rnd_size = min(budget, len(x))
    rnd_idx = np.random.choice(x.index, size=rnd_size, replace=False)
    for c in tqdm(y.columns, 'fitting'):
      model = self.model_class()
      model.fit(x.loc[rnd_idx, :], y[c].loc[rnd_idx])
      self.models.append(model)

  def predict(self, x):
    preds = []
    for model in tqdm(self.models, 'predicting'):
      preds.append(model.predict(x))
    return np.array(preds).T

forecas_classes = [LinearRegression, LGBMRegressor]

ys_hat_te = {}
qs_hat_te = {}
for f in forecas_classes:
  m = Multi_reg(partial(f))
  m.fit(x_tr, y_tr)
  y_hat_cal = m.predict(x_cal)
  ys_hat_te[f.__name__] = m.predict(x_te)
  q_errs = create_pdf(y_cal, y_hat_cal, q_vect = np.arange(11)/10)
  qs_hat_te[f.__name__] = np.expand_dims(ys_hat_te[f.__name__], 2) + np.expand_dims(q_errs, 0)



In [ ]:
# Seasonal AutoETS using direct rolling forward implementation
from statsforecast.models import AutoETS

horizon = y_tr.shape[1]

def rolling_forward_predictions(fitted_model, y_train_hist, y_stream, horizon):
    y_hat = []
    y_train_hist = np.asarray(y_train_hist, dtype=float)
    y_stream = np.asarray(y_stream, dtype=float)

    for i in tqdm(range(len(y_stream)), desc='AutoETS rolling'):
        y_past = np.hstack([y_train_hist, y_stream[:i]])
        preds = fitted_model.forward(y_past, h=horizon)
        y_hat.append(np.asarray(preds['mean'], dtype=float))

    return np.vstack(y_hat)

fitted_model = AutoETS(season_length=24, model='ZZA')
fitted_model.fit(df_tr[target].values)

# Calibration predictions for residual-bootstrap quantiles
y_hat_cal_ets = rolling_forward_predictions(
    fitted_model=fitted_model,
    y_train_hist=df_tr[target].values,
    y_stream=df_cal[target].values,
    horizon=horizon,
)

# Test predictions for model comparison
y_hat_te_ets = rolling_forward_predictions(
    fitted_model=fitted_model,
    y_train_hist=df_tr[target].values,
    y_stream=df_te[target].values,
    horizon=horizon,
)

ys_hat_te['AutoETS_ZZA'] = y_hat_te_ets
q_errs_ets = create_pdf(y_cal.values, y_hat_cal_ets, q_vect=np.arange(11)/10)
qs_hat_te['AutoETS_ZZA'] = np.expand_dims(y_hat_te_ets, 2) + np.expand_dims(q_errs_ets, 0)

print('AutoETS predictions added:', ys_hat_te['AutoETS_ZZA'].shape)



In [ ]:
#@title  Fast Quantile Plot (imported from GitHub)
import importlib.util
from pathlib import Path

import requests

# Keeps the notebook standalone in Colab by downloading the helper module at runtime.
PLOT_UTIL_URL = "https://raw.githubusercontent.com/nepslor/B5203E-TSAF/main/utils/plot_util.py"
PLOT_UTIL_PATH = Path("plot_util.py")

resp = requests.get(PLOT_UTIL_URL, timeout=30)
resp.raise_for_status()
PLOT_UTIL_PATH.write_bytes(resp.content)

spec = importlib.util.spec_from_file_location("plot_util", PLOT_UTIL_PATH)
plot_util = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_util)

qs_animation_plotly = plot_util.qs_animation_plotly


In [ ]:
keys = list(qs_hat_te.keys())
skip = 24*7
n_plots = 400
for k in keys:
  display(qs_animation_plotly(y_te.values[skip:skip+n_plots], ys_hat_te[k][skip:skip+n_plots], qs_hat_te[k][skip:skip+n_plots], n_rows=n_plots, f_name=k))


In [ ]:
%pip install 'chronos-forecasting>=2.1' 'pandas[pyarrow]' 'matplotlib'


In [ ]:
# Chronos input with target only (plus timestamp and item id)
x_te_cr = x_te[[target]].copy().reset_index()
x_te_cr['item_id'] = 'temp'
x_te_cr.head()


In [ ]:
from chronos import BaseChronosPipeline, Chronos2Pipeline
import torch

device_map = "mps" if torch.backends.mps.is_available() else "cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map=device_map)

In [ ]:
# Generate predictions with covariates
def predict_chronos(x_te_cr, n=400, n_context = 24*7):
  cron_qs=[0.01, 0.1, 0.2, 0.5, 0.7, 0.8, 0.9, 0.99]
  w = np.arange(n_context)
  preds = []
  quantiles = []
  for i in tqdm(range(n)):
    df_context = x_te_cr.loc[w+i]
    cron_pred_df = pipeline.predict_df(
        df_context,
        prediction_length=48,
        quantile_levels=cron_qs,
        id_column="item_id",
        timestamp_column="Date Time",
        target=target,
    )
    preds.append(cron_pred_df['predictions'])
    quantiles.append(cron_pred_df[[str(q) for q in cron_qs]].values)
  y_hat_te = pd.concat(preds, axis=1).T
  q_hat_te = np.rollaxis(np.dstack(quantiles), 2)
  return y_hat_te, q_hat_te


def predict_chronos_batched(
    x_te_cr,
    n=400,
    n_context=24*7,
    prediction_length=48,
    batch_windows=32,
    cron_qs=None,
):
    """
    Returns:
      y_hat_te: DataFrame (n, prediction_length)
      q_hat_te: np.ndarray (n, prediction_length, n_quantiles)
    """
    if cron_qs is None:
        cron_qs = [0.01, 0.1, 0.2, 0.5, 0.7, 0.8, 0.9, 0.99]
    q_cols = [str(q) for q in cron_qs]

    all_y = []
    all_q = []

    for start in tqdm(range(0, n, batch_windows)):
        end = min(start + batch_windows, n)

        # Build batch contexts: one item_id per rolling origin
        frames = []
        for i in range(start, end):
            ctx = x_te_cr.iloc[i:i + n_context].copy()
            ctx["item_id"] = f"temp_{i}"
            frames.append(ctx)

        batch_df = pd.concat(frames, ignore_index=True)

        out = pipeline.predict_df(
            batch_df,
            prediction_length=prediction_length,
            quantile_levels=cron_qs,
            id_column="item_id",
            timestamp_column="Date Time",
            target=target,
        )

        # Robust ordering: parse origin index from item_id, then sort by forecast timestamp
        out = out.copy()
        out["origin"] = out["item_id"].str.rsplit("_", n=1).str[-1].astype(int)
        out = out.sort_values(["origin", "Date Time"]).reset_index(drop=True)

        # Build outputs per origin safely (no blind reshape)
        for origin, g in out.groupby("origin", sort=True):
            yv = g["predictions"].to_numpy()
            qv = g[q_cols].to_numpy()

            if len(yv) != prediction_length:
                raise ValueError(
                    f"Origin {origin}: expected {prediction_length} rows, got {len(yv)}"
                )

            all_y.append(yv)
            all_q.append(qv)

    y_hat_te = pd.DataFrame(np.stack(all_y, axis=0))
    q_hat_te = np.stack(all_q, axis=0)

    return y_hat_te, q_hat_te

In [ ]:
n_context = 24*30
n_rows = 24*300

y_hat_te_chronos, q_hat_te_chronos = predict_chronos_batched(x_te_cr, n_context=n_context, n=n_rows, batch_windows=128)
display(qs_animation_plotly(y_te.values[n_context : n_context + n_plots], y_hat_te_chronos.values[:n_plots], q_hat_te_chronos[:n_plots], n_rows=n_plots, f_name='cronos2'))


## Calculate MAE for Chronos2 and other models over the common evaluation period.


The evaluation will be performed over the common evaluation period of `n_rows` (400) samples starting from `n_context` (168) in the test set.


In [ ]:
from sklearn.metrics import mean_absolute_error
get_mae = lambda y, y_hat: np.vstack([mean_absolute_error(y[:, i], y_hat[:, i]) for i in range(y_hat.shape[1])]).ravel()

y_true_common = y_te.values[n_context : n_context + n_rows]

# Store all model predictions to compare in a single container
y_hat_te = {name: pred[n_context : n_context + n_rows] for name, pred in ys_hat_te.items()}
y_hat_te['Chronos2'] = y_hat_te_chronos.to_numpy()

mae_results = {name: get_mae(y_true_common, pred) for name, pred in y_hat_te.items()}
mae_df = pd.DataFrame(mae_results)

print('Mean Absolute Errors for all models over the common evaluation period:')
print(mae_df.head())


In [ ]:
y_te.shape[0] - 24*30

In [ ]:
import matplotlib.pyplot as plt

# Plot the MAE for all models
plt.figure(figsize=(12, 6))
mae_df.plot(ax=plt.gca())
plt.title('Mean Absolute Error across Prediction Horizons for All Models')
plt.xlabel('Prediction Horizon (hours)')
plt.ylabel('MAE')
plt.legend(title='Model')
plt.grid(True)
plt.show()

## Additional comparison: moving-average detrend preprocessing
We subtract a causal moving-average trend from the test target before Chronos and LGMB forecasting, then add back the last available trend level at each forecast origin.


In [ ]:
# Moving-average detrend (causal) + Chronos + inverse transform
ma_window = 24 * 7

trend_te = (df_weather[target].rolling(window=ma_window, min_periods=1).mean().loc[df_te.index].reset_index(drop=True))

x_te_cr_ma = x_te_cr.copy()
x_te_cr_ma[target] = x_te_cr_ma[target].to_numpy() - trend_te.to_numpy()

y_hat_te_ma, q_hat_te_ma = predict_chronos_batched(
    x_te_cr_ma,
    n_context=n_context,
    n=n_rows,
    batch_windows=128,
)

# Re-trend using the last known causal trend level in each context window
origin_idx = np.arange(n_rows) + n_context - 1
trend_level = trend_te.iloc[origin_idx].to_numpy()[:, None]
y_hat_te_ma_retrended = y_hat_te_ma.to_numpy() + trend_level

# LGBM with the same MA(7d) detrend on targets
trend_all = df_weather[target].rolling(window=ma_window, min_periods=1).mean()
horizons = [int(c.rsplit('_', 1)[-1]) * -1 for c in y_tr.columns]  # 1..48

def build_trend_matrix(index, horizons, trend_series):
    mat = np.column_stack([trend_series.shift(-h).loc[index].to_numpy() for h in horizons])
    return mat

trend_tr = build_trend_matrix(y_tr.index, horizons, trend_all)
trend_cal = build_trend_matrix(y_cal.index, horizons, trend_all)
trend_te_mat = build_trend_matrix(y_te.index, horizons, trend_all)

y_tr_ma = y_tr.to_numpy() - trend_tr
y_cal_ma = y_cal.to_numpy() - trend_cal

m_ma = Multi_reg(partial(LGBMRegressor))
m_ma.fit(x_tr, pd.DataFrame(y_tr_ma, index=y_tr.index, columns=y_tr.columns))
y_hat_cal_ma = m_ma.predict(x_cal)
y_hat_te_ma_lgbm = m_ma.predict(x_te)

# Re-trend LGBM predictions back to original units
y_hat_te_ma_lgbm_retrended = y_hat_te_ma_lgbm + trend_te_mat

# Optional calibrated quantiles (kept for consistency with existing setup)
q_errs_ma = create_pdf(y_cal_ma, y_hat_cal_ma, q_vect=np.arange(11)/10)
qs_hat_te_ma_lgbm = np.expand_dims(y_hat_te_ma_lgbm_retrended, 2) + np.expand_dims(q_errs_ma, 0)

# Append MA-detrended variants to the same prediction container
y_hat_te['Chronos2_MA7d_detrend'] = y_hat_te_ma_retrended
y_hat_te['LGBMRegressor_MA7d_detrend'] = y_hat_te_ma_lgbm_retrended[n_context:n_context + n_rows]

mae_results = {name: get_mae(y_true_common, pred) for name, pred in y_hat_te.items()}
mae_df = pd.DataFrame(mae_results)

plt.figure(figsize=(12, 6))
mae_df.plot(ax=plt.gca())
plt.title('Mean Absolute Error across Prediction Horizons (including MA-detrended Chronos and LGBM)')
plt.xlabel('Prediction Horizon (hours)')
plt.ylabel('MAE')
plt.legend(title='Model')
plt.grid(True)
plt.show()

mae_df.tail(2)
